# 02 — SILVER LAYER & CDC | Demo nhanh trên lớp (Task 2)

**Trạng thái:** Initial load và cả 3 CDC (Update, Insert, Schema Evolution) demo đã chạy xong. Notebook này **chỉ đọc**, không append lại CDC, không MERGE và không xóa checkpoint.

**Thứ tự** lọc chuyến đi lỗi → deduplicate `trip_id` → UPDATE/INSERT qua `MERGE INTO` → schema evolution (`surcharge_fee`).

| Mốc đã thực hiện | Silver version | Minh chứng |
|---|---:|---|
| Initial load 12 tháng và dirty file từ Bronze, đã cleaned, deduplicated | **44** | Snapshot Silver trước CDC |
| CDC UPDATE | **45** | `fare_amount`, `tip_amount` đổi trên cùng trip |
| CDC INSERT | **47** | Trip id mới chỉ có sau INSERT |
| Schema Evolution | **48** | Cột `surcharge_fee` xuất hiện |

**Lưu ý:** Version 46 chạy thừa của update bỏ qua

## 0. Khởi tạo — chạy một lần

In [10]:
from pathlib import Path
import sys
from IPython.display import display
import pandas as pd
from delta.tables import DeltaTable
from pyspark.sql import functions as F

ROOT = Path.cwd().resolve()
if not (ROOT / 'src' / 'silver').is_dir():
    ROOT = ROOT.parent
if not (ROOT / 'src' / 'silver').is_dir():
    raise RuntimeError('Mở notebook từ thư mục gốc project hoặc thư mục notebooks/.')
sys.path.insert(0, str(ROOT))
from src.silver.silver_pipeline import make_spark

spark = make_spark('local[2]')
BRONZE = str(ROOT / 'data/bronze/taxi_trips')
SILVER = str(ROOT / 'data/silver/taxi_trips')
REJECTED = str(ROOT / 'data/silver/rejected_records')
AUDIT = str(ROOT / 'data/silver/batch_audit')

INITIAL_VERSION = 44
UPDATE_VERSION = 45
INSERT_VERSION = 47
SCHEMA_VERSION = 48
UPDATE_TRIP_ID = '93184dd0b5a98cc1efe037f60a0ab8f5f90c178bbb12809ac61b991e106c9c05'
INSERT_TRIP_ID = '33f764b320f526509a5ffc671a628fff2361a066ceeae606bb7f39abd528623f'
UPDATE_INGEST_BATCH = 'demo_20260927T080446776977'
INSERT_INGEST_BATCH = 'demo_20260927T085807873071'

def at_version(v):
    return spark.read.format('delta').option('versionAsOf', v).load(SILVER)

def trip_at(v, trip_id):
    frame = at_version(v)
    selected = [c for c in ('trip_id', 'fare_amount', 'tip_amount', 'surcharge_fee',
                            'ingest_batch_id', 'record_source', 'ingested_at') if c in frame.columns]
    rows = (frame.filter(F.col('trip_id') == trip_id)
            .select(*selected).limit(2).collect())
    return [r.asDict(recursive=True) for r in rows]

def display_comparison(rows_before, rows_after, v_before, v_after, trip_id):
    entries = []
    for label, version, rows in [('BEFORE', v_before, rows_before), ('AFTER', v_after, rows_after)]:
        if rows:
            x = rows[0]
            entries.append({'Mốc':label, 'Silver version': version,
                            'trip_id (8 ký tự)':str(x.get('trip_id',''))[:8],
                            'fare_amount':x.get('fare_amount'), 'tip_amount':x.get('tip_amount'),
                            'ingest_batch_id':x.get('ingest_batch_id')})
        else:
            entries.append({'Mốc':label, 'Silver version':version,
                            'trip_id (8 ký tự)':trip_id[:8],
                            'fare_amount':'KHÔNG CÓ', 'tip_amount':'KHÔNG CÓ',
                            'ingest_batch_id':'—'})
    display(pd.DataFrame(entries))

print('Project:', ROOT)
print('Demo versions:', INITIAL_VERSION, UPDATE_VERSION, INSERT_VERSION, SCHEMA_VERSION)

Project: D:\delta-lakehouse-project
Demo versions: 44 45 47 48


## 1. Chỉ ra Silver được hình thành từ Bronze (initial load → v44)

Đọc lịch sử Delta **không phải** đếm lại 12 tháng dữ liệu. Mục đích: chứng minh các giao dịch MERGE đã tạo snapshot; v44 là baseline trước CDC.

In [12]:
silver_delta = DeltaTable.forPath(spark, SILVER)
h = silver_delta.history()
cols = ['version', 'timestamp', 'operation', 'operationMetrics']
# display(h.filter(F.col('version').between(44, 48)).select(*cols).orderBy('version').toPandas())
# CODE MỚI (DÙNG PYSPARK THUẦN)
h.filter(F.col('version').between(44, 48)).select(*cols).orderBy('version').show(truncate=False)
latest = silver_delta.history(1).select('version').first()['version']
print('Silver version hiện tại:', latest)
print('PASS: Có đủ dữ liệu tới v48' if latest >= SCHEMA_VERSION else 'CHƯA ĐỦ: kiểm tra lại đường dẫn/version')

+-------+-----------------------+---------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|version|timestamp              |operation|operationMetrics                                                                                                                                                                                                                                                                                                                                                    

## 2. Cleaning: hợp lệ ở Silver, bản ghi lỗi nằm trong Quarantine

**BEFORE:** Bronze giữ dữ liệu raw (kể cả lỗi). **AFTER:** Silver lọc các dòng không hợp lệ; rejected được giữ riêng cùng `rejection_reasons`.

Để demo nhanh, lấy vài bản ghi mẫu và **không** dùng `.count()` toàn bộ Bronze/Silver ở phần chính.

In [ ]:
bronze = spark.read.format('delta').load(BRONZE)
silver_now = at_version(SCHEMA_VERSION)
rejected = spark.read.format('delta').load(REJECTED)

print('BEFORE — Bronze raw:')
display(bronze.select('trip_id','fare_amount','PULocationID','DOLocationID','record_source').limit(5).show(truncate=False))

print('AFTER — Silver cleansed:')
display(silver_now.select('trip_id','fare_amount','PULocationID','DOLocationID','record_source').limit(5).show(truncate=False))

print('QUARANTINE — ví dụ bản ghi bị loại và lý do:')
display(rejected.select('trip_id','rejection_reasons','record_source').limit(5).show(truncate=False))

BEFORE — Bronze raw:
+----------------------------------------------------------------+-----------+------------+------------+-------------+
|trip_id                                                         |fare_amount|PULocationID|DOLocationID|record_source|
+----------------------------------------------------------------+-----------+------------+------------+-------------+
|57fb6175c2cb9239ff9f745f956a8b11c2361f92ebae8e9b67b892baae497350|42.9       |138         |114         |official_tlc |
|93184dd0b5a98cc1efe037f60a0ab8f5f90c178bbb12809ac61b991e106c9c05|26.8       |93          |157         |official_tlc |
|eb973aab6173a6db90778720a26c63f28c1ecdba189b08a3099d6a743df928eb|19.8       |68          |13          |official_tlc |
|b0602fafe9eb64d6c2031612b04d426b1e6fa1453227f4b07c45631ef8a0459b|17.7       |234         |87          |official_tlc |
|bc69c9ed6977dda1d4e8cdfe6048f37e8469c83c2c45743ccc0f9960916f8cd6|14.9       |230         |151         |official_tlc |
+--------------------------

None

AFTER — Silver cleansed:
+----------------------------------------------------------------+-----------+------------+------------+-------------+
|trip_id                                                         |fare_amount|PULocationID|DOLocationID|record_source|
+----------------------------------------------------------------+-----------+------------+------------+-------------+
|00017f17bd9242a483d85f2785765e8f2e97570f29b62957da166507e7b2fb11|4.4        |125         |158         |official_tlc |
|0002f5560f58e1ccfe18909101093cce30421074dfe7eb4c3f2284d9c069e86f|7.2        |233         |164         |official_tlc |
|000352c0dc0f5ba7891369b5c84e5f5ab7adb5aab02a3a7f712f59f05aea7a94|10.0       |43          |43          |official_tlc |
|0003cab48c71b71d84bacc334f55ad5418e684e20c319b74e7630bd7a14b3052|5.1        |140         |140         |official_tlc |
|000498d6f7fc4c870cbf05fdbbf38599f3f90b4fde24ae07b7489e0323a4e9b8|20.0       |48          |48          |official_tlc |
+----------------------

None

QUARANTINE — ví dụ bản ghi bị loại và lý do:
+----------------------------------------------------------------+------------------------------------+-------------+
|trip_id                                                         |rejection_reasons                   |record_source|
+----------------------------------------------------------------+------------------------------------+-------------+
|096e70371f5d054005b85c16bd5e62242a49f398aa73cc6e4d7c7526bca92446|[invalid_fare]                      |official_tlc |
|e5f334bccbf5376704528d4d33219aebab3d4587ef8f9c67e323448cd4ce7f48|[invalid_fare]                      |official_tlc |
|17a73d051cbbda9800b795302882a348f2255e4e1cc683bfdc2f910e76ec177e|[invalid_fare, invalid_total_amount]|official_tlc |
|3b01d49f307988eb49f7e29b56e2d8abd6102d9e97252db975abd97838d5ba93|[invalid_fare]                      |official_tlc |
|c247f5478076488c9688605a6cd58cd2c3c98a85deadf38e8b46c345a6de7dab|[invalid_fare]                      |official_tlc |
+----------

None

## 3. Deduplication / thống kê microbatch đã lưu

`valid`: dòng qua validation; `winners`: dòng được chọn sau deduplicate **trong một microbatch**. MERGE theo `trip_id` xử lý trùng giữa các microbatch. Bảng `batch_audit` là bằng chứng đã ghi trong lúc pipeline chạy; không bắt notebook tính lại toàn bộ dữ liệu.

In [16]:
audit = spark.read.format('delta').load(AUDIT)
show_cols = [c for c in ('batch_id','incoming','valid','rejected','winners',
                         'silver_version_before','silver_version_after','silver_operation') if c in audit.columns]
display(audit.orderBy(F.desc('batch_id')).select(*show_cols).limit(8).show(truncate=False))
print('Giải thích: incoming = valid + rejected; winners <= valid (dedup theo trip_id trong batch).')

+--------+--------+-------+--------+-------+---------------------+--------------------+----------------+
|batch_id|incoming|valid  |rejected|winners|silver_version_before|silver_version_after|silver_operation|
+--------+--------+-------+--------+-------+---------------------+--------------------+----------------+
|47      |1       |1      |0       |1      |47                   |48                  |MERGE           |
|46      |1       |1      |0       |1      |46                   |47                  |MERGE           |
|45      |1       |1      |0       |1      |45                   |46                  |MERGE           |
|44      |1       |1      |0       |1      |44                   |45                  |MERGE           |
|43      |5218    |3120   |2098    |2992   |43                   |44                  |MERGE           |
|42      |1048576 |1033252|15324   |1033252|42                   |43                  |MERGE           |
|41      |1048576 |1028730|19846   |1028729|41         

None

Giải thích: incoming = valid + rejected; winners <= valid (dedup theo trip_id trong batch).


## 4. CDC UPDATE — **BEFORE v44 → AFTER v45**

Chọn đúng ID đã in ra khi chạy `--mode update`; **không lấy record CDC mới nhất** vì sau UPDATE bạn còn chạy INSERT và schema. So sánh trên **cùng một trip_id**:

- BEFORE: Silver v44, dữ liệu taxi trước CDC.
- AFTER: Silver v45, bản cập nhật fare/tip đã được MERGE.
- Bằng chứng bổ sung: `ingest_batch_id` của AFTER phải là ID UPDATE đã ghi.

In [17]:
update_before = trip_at(INITIAL_VERSION, UPDATE_TRIP_ID)
update_after = trip_at(UPDATE_VERSION, UPDATE_TRIP_ID)
print('CDC UPDATE — cùng trip_id:', UPDATE_TRIP_ID)
display_comparison(update_before, update_after, INITIAL_VERSION, UPDATE_VERSION, UPDATE_TRIP_ID)
if update_before and update_after:
    old, new = update_before[0], update_after[0]
    fare_diff = (new['fare_amount'] or 0) - (old['fare_amount'] or 0)
    tip_diff = (new['tip_amount'] or 0) - (old['tip_amount'] or 0)
    print(f'Delta fare = {fare_diff:+.2f}; Delta tip = {tip_diff:+.2f}')
    print('UPDATE DEMO:', 'PASS' if (fare_diff != 0 or tip_diff != 0)
          and new.get('ingest_batch_id') == UPDATE_INGEST_BATCH else 'KIỂM TRA LẠI DỮ LIỆU/VERSION')
else:
    print('THIẾU RECORD ở một snapshot — kiểm tra đúng Silver path và version.')

CDC UPDATE — cùng trip_id: 93184dd0b5a98cc1efe037f60a0ab8f5f90c178bbb12809ac61b991e106c9c05


,Mốc,Silver version,trip_id (8 ký tự),fare_amount,tip_amount,ingest_batch_id
0,BEFORE,44,93184dd0,26.8,5.86,yellow_tripdata_2025-09
1,AFTER,45,93184dd0,28.8,6.86,demo_20260927T080446776977


Delta fare = +2.00; Delta tip = +1.00
UPDATE DEMO: PASS


## 5. CDC INSERT — **BEFORE v46 → AFTER v47**

Version 46 là snapshot ngay trước version 47. Dùng `new_trip_id` thực tế đã lưu, **khác với source trip_id của UPDATE**.

- BEFORE v46: ID mới **không tồn tại**.
- AFTER v47: ID mới **xuất hiện** → minh họa nhánh `WHEN NOT MATCHED THEN INSERT`.

In [18]:
insert_before = trip_at(INSERT_VERSION - 1, INSERT_TRIP_ID)
insert_after = trip_at(INSERT_VERSION, INSERT_TRIP_ID)
print('CDC INSERT — NEW trip_id:', INSERT_TRIP_ID)
display_comparison(insert_before, insert_after, INSERT_VERSION - 1, INSERT_VERSION, INSERT_TRIP_ID)
print('INSERT DEMO:', 'PASS' if not insert_before and insert_after
      and insert_after[0].get('ingest_batch_id') == INSERT_INGEST_BATCH else
      'KIỂM TRA LẠI ID/VERSION/INGEST BATCH')

CDC INSERT — NEW trip_id: 33f764b320f526509a5ffc671a628fff2361a066ceeae606bb7f39abd528623f


,Mốc,Silver version,trip_id (8 ký tự),fare_amount,tip_amount,ingest_batch_id
0,BEFORE,46,33f764b3,KHÔNG CÓ,KHÔNG CÓ,—
1,AFTER,47,33f764b3,28.8,6.86,demo_20260927T085807873071


INSERT DEMO: PASS


## 6. Schema Evolution — **BEFORE v47 → AFTER v48**

`--mode schema` thêm `surcharge_fee=1.5` vào Bronze; MERGE với schema evolution đưa cột mới sang Silver. So sánh metadata **hai snapshot**, sau đó lấy một dòng có giá trị ở cột mới.

In [19]:
schema_before = at_version(INSERT_VERSION)
schema_after = at_version(SCHEMA_VERSION)
old_cols, new_cols = set(schema_before.columns), set(schema_after.columns)
display(pd.DataFrame([
    {'Mốc':'BEFORE', 'Silver version':INSERT_VERSION, 'có surcharge_fee?':'surcharge_fee' in old_cols},
    {'Mốc':'AFTER', 'Silver version':SCHEMA_VERSION, 'có surcharge_fee?':'surcharge_fee' in new_cols},
]))
print('Các cột mới ở v48:', sorted(new_cols - old_cols))
print('Schema field:', schema_after.schema['surcharge_fee'] if 'surcharge_fee' in new_cols else 'CHƯA CÓ')
if 'surcharge_fee' in new_cols:
    cols = ['trip_id','surcharge_fee','ingest_batch_id']
    demo_rows = (schema_after.filter(F.col('surcharge_fee').isNotNull())
                 .select(*cols).limit(3).collect())
    display(pd.DataFrame([r.asDict() for r in demo_rows]))
    print('SCHEMA DEMO:', 'PASS' if 'surcharge_fee' not in old_cols and demo_rows
          and any(float(r['surcharge_fee']) == 1.5 for r in demo_rows) else 'KIỂM TRA LẠI')

,Mốc,Silver version,có surcharge_fee?
0,BEFORE,47,False
1,AFTER,48,True


Các cột mới ở v48: ['surcharge_fee']
Schema field: StructField('surcharge_fee', DoubleType(), True)


,trip_id,surcharge_fee,ingest_batch_id
0,93184dd0b5a98cc1efe037f60a0ab8f5f90c178bbb1280...,1.5,demo_20260927T091027666122


SCHEMA DEMO: PASS


## 7. TÙY CHỌN — Full-table quality check (KHÔNG chạy lúc demo nhanh)

Bật cell này trước buổi học nếu cần số liệu tổng: `count()` trên hàng chục triệu dòng sẽ tốn thời gian. Trong demo chính, audit và ví dụ BEFORE/AFTER đã đủ minh họa.

In [20]:
RUN_FULL_SCAN = False  # Đổi thành True chỉ khi muốn quét toàn bộ Silver
if RUN_FULL_SCAN:
    total = silver_now.count()
    invalid = silver_now.filter(
        F.col('fare_amount').isNull() | (F.col('fare_amount') <= 0) |
        F.col('PULocationID').isNull() | F.col('DOLocationID').isNull()
    ).count()
    duplicate_keys = silver_now.groupBy('trip_id').count().filter(F.col('count') > 1).count()
    display(pd.DataFrame([{'silver_rows':total,'invalid_main_rules':invalid,'duplicate_trip_ids':duplicate_keys}]))
else:
    print('Bỏ qua full scan để tiết kiệm thời gian. Bật RUN_FULL_SCAN=True nếu cần.')

Bỏ qua full scan để tiết kiệm thời gian. Bật RUN_FULL_SCAN=True nếu cần.


## Chốt Task 2 / Chuyển sang Task 3

**Kết luận nên nói:** Initial v44 đã lọc/dedup từ Bronze; v45 UPDATE cùng ID; v47 INSERT ID mới; v48 tiếp nhận `surcharge_fee`. Task 3 sẽ giải thích **Delta lưu v44/v45 thế nào** bằng `versionAsOf`, History và `_delta_log`.

Notebook này **không chạy pipeline hoặc demo CDC lần nữa**. Nếu muốn giải phóng RAM sau cả hai notebook, dùng cell stop ở cuối notebook 03.